# Evaluation Metrics for Generative Modeling on Spatial Transcriptomics
**Data:** paired spatial-transcriptomics slides (Zhuang ABCA mouse-brain atlas).

A runnable companion to the `paired_slides_eval` suite. The notebook follows one linear workflow:

1. **§1 Generate** the cells to evaluate (per model → a `generated.h5ad`).
2. **§2 Build the shared apparatus** — load the **source / target / classifier** slides and fit
   them into *one* whitened-PCA(50) basis + standardised-coord frame (the NicheFlow recipe).
3. **§3 Train probes** — the cell-type classifier for `ct/*`, and the masked expression regressor
   for `recon/*`.
4. **§4 Evaluate** — run the whole suite and write per-model CSVs (**§4a**), or run a single metric
   group and just print it (**§4b**).

Everything is scored through the **same** apparatus, so metrics are comparable across models: a
model that emits gene-space cells + raw coords (OT-CFM) is projected/standardised into the exact
basis the niche models (NicheFlow CFM/VFM) already live in — handled automatically by
`evaluate_files(..., coords="auto")`. All paths below are **relative to `notebooks/`**.

## 0. Environment setup

Per-project `uv` venv (run from the `eval_metrics/` repo root):

In [15]:
%%bash
UV_LINK_MODE=copy uv sync --extra pipeline --extra nicheflow --extra classifier 

Resolved 268 packages in 1ms
Checked 245 packages in 96ms


In [16]:
%%bash
cd /work/magroup/shared/enfuzers/generative_modeling/eval_metrics
# env -u VIRTUAL_ENV uv run python -m ipykernel install --user \
#     --name eval_metrics --display-name "Python (eval_metrics)"

In [2]:
import sys, torch
print(sys.executable)
print(torch.__version__, torch.version.cuda)

/work/magroup/shared/enfuzers/generative_modeling/eval_metrics/.venv/bin/python3
2.12.1+cu126 12.6


In [1]:
import importlib

import numpy as np
import pandas as pd


def _has(mod):
    try:
        importlib.import_module(mod)
        return True
    except Exception:
        return False


print("optional backends present:")
print("  torch   (MMD / EMD / classifier):", _has("torch"))
print("  ot/pot  (EMD / Wasserstein)     :", _has("ot"))
print("  squidpy (Moran's I)             :", _has("squidpy"))

optional backends present:
  torch   (MMD / EMD / classifier): True
  ot/pot  (EMD / Wasserstein)     : True
  squidpy (Moran's I)             : False


## 1. Generate the cells to evaluate

The metrics compare the real **target slide** against a model's **generated cells**. Generation is
model-specific and already done for the four models below — each stores its cells as a flat/niche
`.h5ad`. The cell just confirms the files are present.

| key | emits | needs shared-PCA replay? |
|---|---|---|
| `otcfm` / `otcfm_spatial` | flat **gene-space** cells + **raw** coords | yes (`shared_pca` + `standardize_coords`) |
| `nicheflow_cfm` / `nicheflow_vfm` | already **whitened X_pca(50)** + **standardised** coords | no (pass-through) |

The next cell just checks the files exist; the one after it **regenerates** a model on the
allocated GPU (run it only if you want fresh cells — it overwrites and needs a checkpoint).

In [ ]:
import os

# Each model's generated cells (one flat/niche .h5ad per model).
ARTIFACTS = {
    "otcfm":          "../artifacts/otcfm_1025/generated.h5ad",               # gene-space + raw coords
    "otcfm_spatial":  "../artifacts/otcfm_spatial_1025/generated.h5ad",       # gene-space + raw coords
    "nicheflow_cfm":  "../artifacts/nicheflow_cfm_unaligned/generated.h5ad",  # whitened X_pca + std coords
    "nicheflow_vfm":  "../artifacts/nicheflow_vfm/generated.h5ad",            # whitened X_pca + std coords
}
for name, path in ARTIFACTS.items():
    print(f"{'OK     ' if os.path.exists(path) else 'MISSING'}  {name:14s} {path}")

In [19]:
%%bash
# OPTIONAL — (re)generate NicheFlow CFM cells on the allocated GPU (overwrites the artifact; needs a
# checkpoint).
cd ..
# uv run python -m paired_slides_eval.generate generator=nicheflow \
#     source=../nicheflow_mba/data/adata_Zhuang_Zhuang-ABCA-1.024.h5ad \
#     target=../nicheflow_mba/data/adata_Zhuang_Zhuang-ABCA-1.025.h5ad \
#     checkpoint=ckpts/NicheFlow_CFM_ABCA.ckpt \
#     generated_out=artifacts/nicheflow_cfm_unaligned/generated.h5ad
../nicheflow_mba/.venv/bin/python -m paired_slides_eval.generate generator=nicheflow \
      source=../nicheflow_mba/data/adata_Zhuang_Zhuang-ABCA-1.024.h5ad \
      target=../nicheflow_mba/data/adata_Zhuang_Zhuang-ABCA-1.025.h5ad \
      checkpoint=../nicheflow_mba/ckpts/NicheFlow_CFM_ABCA.ckpt \
      generated_out=artifacts/nicheflow_cfm_unaligned/generated.h5ad

[14:27:51][paired_slides_eval.adapters.nicheflow.h5ad_preprocessor][INFO] Preprocessing the spatial coordinates per timepoint with: standardization
[14:27:51][paired_slides_eval.adapters.nicheflow.h5ad_preprocessor][WARNING] Using `CPU`! Might be too slow!


Computing radius graphs for timepoint: 'B': 100%|██████████| 2/2 [00:06<00:00,  3.47s/it]
Subsampling centroids t='B' | dx=0.15 | dy=0.2: 100%|██████████| 2/2 [00:00<00:00, 37.68it/s]


[14:27:58][paired_slides_eval.adapters.nicheflow.h5ad_preprocessor][INFO] Fixing test microenvironments to 513 per slice.
[14:28:05][nicheflow.datasets.st_dataset_base][INFO] [rank: 0] Creating per timepoint PyTorch Geoemtric Data objects


/work/magroup/shared/enfuzers/generative_modeling/nicheflow_mba/.venv/lib/python3.12/site-packages/torchdyn/numerics/odeint.py:83: UserWarning: Setting tolerances has no effect on fixed-step methods
  warn("Setting tolerances has no effect on fixed-step methods")


Your vector field does not have `nn.Parameters` to optimize.
generated: 513 niches x 68 points -> artifacts/nicheflow_cfm_unaligned/generated.h5ad


In [21]:
%%bash
# Generate Nicheflow VFM cells 
cd ..
../nicheflow_mba/.venv/bin/python -m paired_slides_eval.generate generator=nicheflow \
      generator.variant=vfm \
      source=../nicheflow_mba/data/adata_Zhuang_Zhuang-ABCA-1.024.h5ad \
      target=../nicheflow_mba/data/adata_Zhuang_Zhuang-ABCA-1.025.h5ad \
      checkpoint=../nicheflow_mba/ckpts/NicheFlow_VFM_ABCA.ckpt \
      generated_out=artifacts/nicheflow_vfm_unaligned/generated.h5ad

[14:42:44][paired_slides_eval.adapters.nicheflow.h5ad_preprocessor][INFO] Preprocessing the spatial coordinates per timepoint with: standardization
[14:42:44][paired_slides_eval.adapters.nicheflow.h5ad_preprocessor][WARNING] Using `CPU`! Might be too slow!


Computing radius graphs for timepoint: 'B': 100%|██████████| 2/2 [00:06<00:00,  3.42s/it]
Subsampling centroids t='B' | dx=0.15 | dy=0.2: 100%|██████████| 2/2 [00:00<00:00, 37.88it/s]


[14:42:51][paired_slides_eval.adapters.nicheflow.h5ad_preprocessor][INFO] Fixing test microenvironments to 513 per slice.
[14:42:58][nicheflow.datasets.st_dataset_base][INFO] [rank: 0] Creating per timepoint PyTorch Geoemtric Data objects


/work/magroup/shared/enfuzers/generative_modeling/nicheflow_mba/.venv/lib/python3.12/site-packages/torchdyn/numerics/odeint.py:83: UserWarning: Setting tolerances has no effect on fixed-step methods
  warn("Setting tolerances has no effect on fixed-step methods")


Your vector field does not have `nn.Parameters` to optimize.
generated: 513 niches x 68 points -> artifacts/nicheflow_vfm_unaligned/generated.h5ad


In [24]:
%%bash
# OPTIONAL — (re)generate the OT-CFM cells (needs fm_mnist importable + its checkpoints). Run this
# after the PCA-space change so the artifacts are emitted in the model's OWN whitened-PCA space
# (carrying the PCA→gene inverse in uns['source_pca']); older artifacts stored gene-space raw counts.
# The eval command is unchanged — width auto-detection inverts + reprojects into the shared basis.
cd ..
FM=../fm_mnist                    # so `import fm` resolves (generator.fm_root)
DATA=../nicheflow_mba/data

uv run python -m paired_slides_eval.generate generator=otcfm \
    target=$DATA/adata_Zhuang_Zhuang-ABCA-1.025.h5ad \
    source=null \
    checkpoint=$FM/outputs/cfm_mouse_pca5_1025/ckpt_final.pt \
    generator.fm_root=$FM \
    generated_out=artifacts/otcfm_1025/generated.h5ad

uv run python -m paired_slides_eval.generate generator=otcfm_spatial \
    source=$DATA/adata_Zhuang_Zhuang-ABCA-1.025.h5ad \
    target=$DATA/adata_Zhuang_Zhuang-ABCA-1.025.h5ad \
    checkpoint=$FM/outputs/cfm_mouse_pca5_1025_spatial/ckpt_final.pt \
    generator.fm_root=$FM \
    generated_out=artifacts/otcfm_spatial_1025/generated.h5ad

generated: 9962 cells, 5 feats (flat slide) -> artifacts/otcfm_1025/generated.h5ad


generated: 9962 cells, 5 feats (flat slide) -> artifacts/otcfm_spatial_1025/generated.h5ad


## 2. Build the shared evaluation apparatus (source · target · classifier slide)

One `preprocess_pair` fit defines the whole measuring apparatus and is reused everywhere:

- **Expression:** `normalize_total → log1p → shared PCA (fit on source+target) → whiten`.
- **Coordinates:** per-slide standardisation `(pos − mean) / std`.

It produces two pickles that carry the *persisted recipe* (`lognorm_mean`, `lognorm_target_sum`,
`var_names`, `PCs`, `stats['X_pca']`, `stats['coords']`):

| file | what it is |
|---|---|
| `../data/abca_pair.pkl` | the apparatus **and** the eval target — source `1.024` → target `1.025` |
| `../data/abca_1.026_clf.pkl` | the held-out **classifier** slide `1.026` projected into the *same* basis |

Slide identities for this dataset: **target = `1.025`** (what every model is scored against),
**generation pair = `1.024 → 1.025`**, **classifier slide = `1.026`** (held out — neither source
nor target). Equivalent one-liner CLI:

In [25]:
%%bash
uv run python -m paired_slides_eval.adapters.prepare_shared_slides \
  --source ../../nicheflow_mba/data/adata_Zhuang_Zhuang-ABCA-1.024.h5ad \
  --target ../../nicheflow_mba/data/adata_Zhuang_Zhuang-ABCA-1.025.h5ad \
  --classifier_slide ../../nicheflow_mba/data/adata_Zhuang_Zhuang-ABCA-1.026.h5ad \
  --ct_key class --n_pcs 50 \
  --out_pair ../data/abca_pair.pkl --out_classifier ../data/abca_1.026_clf.pkl

Using `CPU`! Might be too slow!
Computing radius graphs for timepoint: 'B': 100%|██████████| 2/2 [00:06<00:00,  3.44s/it]
Subsampling centroids t='B' | dx=0.15 | dy=0.2: 100%|██████████| 2/2 [00:00<00:00, 31.28it/s]
Using `CPU`! Might be too slow!
Computing radius graphs for timepoint: 'C': 100%|██████████| 1/1 [00:02<00:00,  2.93s/it]
Subsampling centroids t='C' | dx=0.15 | dy=0.2:   0%|          | 0/1 [00:00<?, ?it/s]You should change the values for `dx` and `dy`.GT: 9134 | Microenvironment cover: 9111
Subsampling centroids t='C' | dx=0.15 | dy=0.2: 100%|██████████| 1/1 [00:00<00:00, 27.31it/s]


saved shared pair -> ../data/abca_pair.pkl (target timepoint=B, n_pcs=50)
saved classifier slide -> ../data/abca_1.026_clf.pkl (9134 cells, 20 classes)


In [29]:
import pickle

ds = pickle.load(open("../data/abca_pair.pkl", "rb"))
print("headline neutral_k =", ds.neutral_k)          # the k eval uses by default
print("P* stored at        =", ds.neutral_pcs.shape[1], "comps (sliced to neutral_k at eval)")


headline neutral_k = 8
P* stored at        = 50 comps (sliced to neutral_k at eval)


In [ ]:
import os
import pickle

from paired_slides_eval.adapters.nicheflow.preprocess import (
    SLIDE_B, preprocess_classifier_slide, preprocess_pair,
)
from paired_slides_eval.data.neutral_basis import attach_neutral_basis, fit_neutral_basis

ds_pair, pre = preprocess_pair(SOURCE_H5AD, TARGET_H5AD, n_pcs=N_PCS, cell_type_column=CT_KEY)

# neutral basis P* (fit on the target's log-gene) — the step that used to live in preprocess_pair
basis, pair_scores = fit_neutral_basis(
    pre._log_gene, pre.timepoint_indices[SLIDE_B],
    n_pcs=N_PCS, target_sum=ds_pair.lognorm_target_sum, var_names=ds_pair.var_names,
)
attach_neutral_basis(ds_pair, basis, pair_scores)

clf_ds = preprocess_classifier_slide(
    CLASSIFIER_H5AD, pre, cell_type_column=CT_KEY, neutral_basis=basis,   # <- new
)
pickle.dump(ds_pair, open(PAIR_PKL, "wb"))
pickle.dump(clf_ds, open(CLF_PKL, "wb"))
print(f"built {PAIR_PKL} (neutral_k={ds_pair.neutral_k})")


## 2b. Train OT-CFM in the shared PCA space (so all models are trained in one basis)

For the metrics to compare models fairly, every model must be **trained** in the *same* whitened-PCA(50)
basis — reconciling *after* generation is lossy (a low-`n_pcs` OT-CFM becomes rank-deficient once
reprojected; see the recon/`c2st` discussion). **NicheFlow already trains in the pair PCA** — its native
basis *is* the shared basis, so it needs no change. **OT-CFM** by default fits its *own* PCA, so we inject
NicheFlow's basis into `fm_mnist`'s preprocessing via `--pca_stats` (`fm.data.load_spatial_pca(external_stats=…)`).

**Order:** §2 must run first (it builds `../data/abca_pair.pkl`, the shared basis). Then **export** that
basis → **retrain** OT-CFM on it → point §1's OT-CFM generation cell at the new checkpoint
(`…/ckpt_final.pt`) and re-run it. The regenerated cells are then 50-d in NicheFlow's exact space, so the
eval reads them straight through — no inversion, no rank deficiency.

In [ ]:
%%bash
# Export §2's shared PCA as an fm_mnist load_spatial_pca stats .npz (CPU, light; eval venv).
cd ..
uv run python -m paired_slides_eval.adapters.otcfm_export \
    --pair data/abca_pair.pkl --out data/shared_pca.npz

In [ ]:
%%bash
# Retrain OT-CFM IN the shared 50-d basis (gene-only + naive-spatial). Uses the root venv
# (.venv: torch + torchcfm + scanpy + sklearn); train_cfm_spatial.py self-adds fm to sys.path.
# Needs a GPU allocation. Keep your usual hyperparameters (--steps/--batch/--lr/--coupling/…);
# --n_pcs/--whiten/--target_sum are ignored once --pca_stats is given (dim auto = 50, or 52 with coords).
cd ../..                                     # -> generative_modeling/
PY=.venv/bin/python
DATA=nicheflow_mba/data
STATS=eval_metrics/data/shared_pca.npz

# gene-only  (-> feeds artifacts/otcfm_1025)
$PY fm_mnist/scripts/train_cfm_spatial.py \
    --data $DATA/adata_Zhuang_Zhuang-ABCA-1.025.h5ad \
    --pca_stats $STATS \
    --out fm_mnist/outputs/cfm_shared50_1025

# naive-spatial  (-> feeds artifacts/otcfm_spatial_1025)
$PY fm_mnist/scripts/train_cfm_spatial.py \
    --data $DATA/adata_Zhuang_Zhuang-ABCA-1.025.h5ad \
    --pca_stats $STATS --spatial_key spatial \
    --out fm_mnist/outputs/cfm_spatial_shared50_1025

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/qimow/.netrc.
wandb: Currently logged in as: qimow (qimow-carnegie-mellon-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.27.0
wandb: Run data is saved locally in /work/magroup/shared/enfuzers/generative_modeling/wandb/run-20260702_164544-jngxrfni
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run dandy-bee-13
wandb: ⭐️ View project at https://wandb.ai/qimow-carnegie-mellon-university/fm-spatial
wandb: 🚀 View run at https://wandb.ai/qimow-carnegie-mellon-university/fm-spatial/runs/jngxrfni


train batches/epoch=35  test cells=996  PCA dim=50


loss=0.8162:  38%|███▊      | 3070/8000 [01:02<01:33, 52.91it/s]

## 3. Train the local-neighbourhood probes *(only if you want `ct/*` and `recon/*`)*

The label-free metrics (§4) run without trained probes. The `concordance` / `ct_gap` groups need one
**`SpatialCTClassifierNet`**, trained on the classifier slide from §2 (`../data/abca_1.026_clf.pkl`)
so it lives in the same basis as the apparatus. This pair has **20** cell types, so override
`data.n_classes=20` (the config default is 17). The probes are also scored in the neutral basis P*
sliced to the pickle's headline **`neutral_k`** (the §2 print shows **8**), not the full 50, so
override `data.n_pcs=8` too — that sets the nets' `input_dim` (and the regressor's `output_dim`) to
match the 8-dim features the dataset feeds. With the default `n_pcs=50` training dies with
`mat1 and mat2 shapes cannot be multiplied (Nx8 and 50x64)`. Train once and reuse the single
checkpoint for every model; the old per-model checkpoints are **incompatible** (different recipe /
vocabulary).

The new `recon/*` group needs a second checkpoint: a masked-centroid expression regressor. It reuses
`SpatialCTClassifierNet` with `output_dim = n_pcs`, but its dataset target is `target="expr"`, so it
predicts the centroid expression from its KNN neighbours. The checkpoint below is copied to
`../ckpts/recon_spatial.ckpt`, and §4 passes it with `--regressor`.

The cell below trains both probes on the **allocated GPU** — the kernel runs inside your SLURM job, so
`experiment=classifier/abca_spatial` / `experiment=classifier/abca_spatial_regressor` land on the
job's GPU automatically.

> **Driver note:** this needs `torch+cu126` (the node's driver is CUDA 12.7). With the current
> `torch+cu130` build it errors with *"driver too old"* — repin to cu126 and `uv sync` first.

In [30]:
%%bash
# Runs on the job's GPU (kernel is inside the SLURM allocation). `cd ..` -> repo root so $PWD and the
# relative out-dirs match a terminal there.
cd ..

# The probes are scored in the neutral basis P* sliced to the pickle's headline `neutral_k` (see the
# §2 print: currently 8), NOT the full n_pcs=50. The nets' input_dim (and the regressor's output_dim)
# come from data.n_pcs, so override it to neutral_k=8 to match the 8-dim features the dataset feeds;
# leaving the default 50 raises "mat1 and mat2 shapes cannot be multiplied (Nx8 and 50x64)".

# Cell-type classifier checkpoint (monitors val/f1) -> ckpts/clf_spatial.ckpt.
uv run python -m paired_slides_eval.classifier.train experiment=classifier/abca_spatial \
    data.datamodule.data_fp=$PWD/data/abca_1.026_clf.pkl data.n_classes=20 data.n_pcs=8 \
    logger=tensorboard hydra.run.dir=outputs/clf_gene_train

# Masked-centroid expression regressor checkpoint (monitors val/mse) -> ckpts/recon_spatial.ckpt.
uv run python -m paired_slides_eval.classifier.train experiment=classifier/abca_spatial_regressor \
    data.datamodule.data_fp=$PWD/data/abca_1.026_clf.pkl data.n_classes=20 data.n_pcs=8 \
    logger=tensorboard hydra.run.dir=outputs/recon_train

task_name: train                                                                
train: true                                                                     
test: true                                                                      
ckpt_path: null                                                                 
seed: '2025'                                                                    
data:                                                                           
  datamodule:                                                                   
    _target_: paired_slides_eval.classifier.datamodule.H5ADCTDataModule         
    split_seed: '2025'                                                          
    train_batch_size: 1024                                                      
    eval_batch_size: 1024                                                       
    num_workers: 2                                                              
    train_ratio: 0.8        

/work/magroup/shared/enfuzers/generative_modeling/eval_metrics/.venv/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python3 /work/magroup/shared/enfuzers/generative_modeling/e ...
Using bfloat16 Automatic Mixed Precision (AMP)
Trainer already configured with model summary callbacks: [<class 'lightning.pytorch.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


[15:07:39][__main__][INFO] [rank: 0] Logging hyperparameters!
[15:07:39][__main__][INFO] [rank: 0] Starting training!


/work/magroup/shared/enfuzers/generative_modeling/eval_metrics/.venv/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python3 /work/magroup/shared/enfuzers/generative_modeling/e ...
/work/magroup/shared/enfuzers/generative_modeling/eval_metrics/.venv/lib/python3.12/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /work/magroup/shared/enfuzers/generative_modeling/eval_metrics/outputs/clf_gene_train/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/work/magroup/shared/enfuzers/generative_modeling/eval_metrics/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision bf16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using

┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                              ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ net                               │ SpatialCT… │ 56.8 K │ train │     0 │
│ 1  │ net.point_proj                    │ Linear     │    576 │ train │     0 │
│ 2  │ net.encoder                       │ Sequential │ 33.8 K │ train │     0 │
│ 3  │ net.encoder.0                     │ SAB        │ 16.9 K │ train │     0 │
│ 4  │ net.encoder.0.mab                 │ MAB        │ 16.9 K │ train │     0 │
│ 5  │ net.encoder.0.mab.fc_q            │ Linear     │  4.2 K │ train │     0 │
│ 6  │ net.encoder.0.mab.fc_k            │ Linear     │  4.2 K │ train │     0 │
│ 7  │ net.encoder.0.mab.fc_v            │ Linear     │  4.2 K │ train │     0 │
│ 8  │ net.encoder.0.mab.ln0             │ LayerNorm  │    128 │ train │     0 │
│ 9  │ net.encoder.0.mab.ln1

...

/work/magroup/shared/enfuzers/generative_modeling/eval_metrics/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


/work/magroup/shared/enfuzers/generative_modeling/eval_metrics/.venv/lib/python3.12/site-packages/lightning/pytorch/loops/fit_loop.py:321: The number of training batches (8) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Epoch 0: 100%|██████████| 8/8 [00:00<00:00, 11.29it/s, v_num=2, train/loss=3.820]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 1: 100%|██████████| 8/8 [00:00<00:00, 32.23it/s, v_num=2, train/loss=3.140, val/mse=3.460]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 2: 100%|██████████| 8/8 [00:00<00:00, 32.75it/s, v_num=2, train/loss=2.860, val/mse=2.980]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 3: 100%|██████████| 8/8 [00:00<00:00, 32.00it/s, v_num=2, train/loss=2.520, val/mse=2.710]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 4: 100%|██████████| 8/8 [00:00<00:00, 32.59it/s, v_num=2, train/loss=2.430, val/mse=2.570]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 5: 100%|██████████| 8/8 [00:00<00:00, 32.67it/s, v_num=2, train/

Restoring states from the checkpoint path at /work/magroup/shared/enfuzers/generative_modeling/eval_metrics/outputs/recon_train/checkpoints/epoch_030.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at /work/magroup/shared/enfuzers/generative_modeling/eval_metrics/outputs/recon_train/checkpoints/epoch_030.ckpt


Testing DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 88.93it/s] 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test/mse          │    2.6484525203704834     │
└───────────────────────────┴───────────────────────────┘
[15:09:38][__main__][INFO] [rank: 0] Best ckpt path: /work/magroup/shared/enfuzers/generative_modeling/eval_metrics/outputs/recon_train/checkpoints/epoch_030.ckpt
[15:09:38][__main__][INFO] [rank: 0] {'lr-AdamW': tensor(0.0004), 'train/loss': tensor(2.2104), 'val/mse': tensor(2.6117), 'test/mse': tensor(2.6485)}


In [31]:
%%bash
cd ../
mkdir -p ckpts
cp "$(ls -t outputs/clf_gene_train/checkpoints/epoch_*.ckpt | head -1)" ckpts/clf_gene.ckpt
cp "$(ls -t outputs/recon_train/checkpoints/epoch_*.ckpt | head -1)" ckpts/recon_spatial.ckpt
echo wrote ckpts/clf_gene.ckpt
echo wrote ckpts/recon_spatial.ckpt

wrote ckpts/clf_gene.ckpt
wrote ckpts/recon_spatial.ckpt


## 4. Run the evaluation

This is the same `python -m paired_slides_eval.evaluate` command you'd run in the terminal (a thin
wrapper over `evaluate_files`). Point every model at the **same** `data/abca_pair.pkl`; reconciliation
into the shared space is automatic, so **one command works for all four models**:

- **Expression** — gene-space cells (OT-CFM) are projected into the pickle's whitened shared PCA;
  already-reduced cells (NicheFlow) pass through (dimension auto-detect).
- **Coordinates** — `--coords auto` (default) detects raw vs already-standardised coords and maps
  them into the target frame, logging the decision. No per-model flags.

`--classifier` enables the `ct/*` groups; `--ct_real_reference fixed` makes `ct/acc_real` one
model-independent constant across the table. `--regressor` enables the `recon/*` group; by default
`recon/mse_real` also uses a fixed seeded real-target reference. The `%%bash` cells `cd ..` to the repo
root so the paths match a terminal there.

### 4a. Full metric suite → per-model CSV

Runs the whole suite for each model and writes `reports/<model>/metrics.csv` (the `--out` `metric,value`
layout). The `ct/*` groups are added when the §3 classifier exists, `recon/*` is added when the §3
regressor exists, and either probe skips cleanly otherwise.

In [32]:
%%bash
cd ..
CKPT=ckpts/clf_gene.ckpt
RECON=ckpts/recon_spatial.ckpt
CLF=""; [ -f "$CKPT" ] && CLF="--classifier $CKPT --ct_real_reference fixed"
REG=""; [ -f "$RECON" ] && REG="--regressor $RECON"
[ -z "$CLF" ] && echo "no classifier at $CKPT -> ct/* groups will skip"
[ -z "$REG" ] && echo "no regressor at $RECON -> recon/* group will skip"
METRIC_GROUPS="psd spd distribution c2st c2st_nn moran concordance ct_gap recon"
for m in otcfm_1025 otcfm_spatial_1025 nicheflow_cfm_unaligned nicheflow_vfm_unaligned; do
    echo "===== $m ====="
    uv run python -m paired_slides_eval.evaluate \
        --target data/abca_pair.pkl \
        --generated artifacts/$m/generated.h5ad \
        --ct_key class --groups $METRIC_GROUPS $CLF $REG \
        --out reports/$m/metrics_npc8.csv
done

===== otcfm_1025 =====


<frozen runpy>:128: RuntimeWarning: 'paired_slides_eval.evaluate' found in sys.modules after import of package 'paired_slides_eval', but prior to execution of 'paired_slides_eval.evaluate'; this may result in unpredictable behaviour
/work/magroup/shared/enfuzers/generative_modeling/eval_metrics/.venv/lib/python3.12/site-packages/docrep/decorators.py:43: SyntaxWarning: 'n_jobs' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)
/work/magroup/shared/enfuzers/generative_modeling/eval_metrics/.venv/lib/python3.12/site-packages/docrep/decorators.py:43: SyntaxWarning: 'show_progress_bar' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)
/work/magroup/shared/enfuzers/generative_modeling/eval_metrics/src/paired_slides_eval/metrics/morans.py:43: FutureWarning: Calling `spatial_neighbors` is deprecated and will be removed in squidpy v1.9.0. Use `spatial_neighbors_knn`, `spatial_neighbors_radius`, `spatial_neighbors_delaunay`, `spatial_neighbor

INFO     Creating graph using `None` transform and `1` libraries.               


/work/magroup/shared/enfuzers/generative_modeling/eval_metrics/src/paired_slides_eval/metrics/morans.py:43: FutureWarning: Calling `spatial_neighbors` is deprecated and will be removed in squidpy v1.9.0. Use `spatial_neighbors_knn`, `spatial_neighbors_radius`, `spatial_neighbors_delaunay`, `spatial_neighbors_grid`, or `spatial_neighbors_from_builder` instead.
  sq.gr.spatial_neighbors(adata, coord_type="generic", n_neighs=n_neighs)


INFO     Creating graph using `None` transform and `1` libraries.               
target: 9962 cells, 8 features
generated: 9962 cells, 8 feats (flat slide)
test/c2st/acc            0.9962
test/c2st/auc            0.9994
test/c2st/gene_acc       0.9965
test/c2st/gene_auc       0.9999
test/c2st/nn             0.0080
test/c2st/nn_flip        0.0000
test/c2st/nn_real_ref    0.9628
test/c2st/nn_std         0.0284
test/c2st/pos_acc        0.6315
test/ct/acc              0.4920
test/ct/acc_gap          0.3509
test/ct/acc_gen          0.3301
test/ct/acc_real         0.6810
test/ct/f1               0.3496
test/ct/prop_jsd         0.1453
test/ct/prop_kl          1.0145
test/ct/prop_tv          0.4441
test/mmd2/pos            0.0661
test/mmd2/x              0.0171
test/moran/corr          0.5143
test/moran/gen_mean      -0.0001
test/moran/mae           0.5115
test/moran/real_mean     0.5114
test/ot_w1/pos           0.4311
test/ot_w1/x             2.6123
test/ot_w2/pos           0.5063
test/ot_w2/

...

INFO     Creating graph using `None` transform and `1` libraries.               
target: 9962 cells, 8 features
generated: 513 niches x 68 points
test/c2st/acc            0.9795
test/c2st/auc            0.9977
test/c2st/gene_acc       0.9727
test/c2st/gene_auc       0.9972
test/c2st/nn             0.0374
test/c2st/nn_flip        0.0000
test/c2st/nn_real_ref    0.9759
test/c2st/nn_std         0.0676
test/c2st/pos_acc        0.7027
test/ct/acc              0.3782
test/ct/acc_gap          0.3769
test/ct/acc_gen          0.3041
test/ct/acc_real         0.6810
test/ct/f1               0.3408
test/ct/prop_jsd         0.0731
test/ct/prop_kl          1.0790
test/ct/prop_tv          0.2066
test/mmd2/pos            0.0175
test/mmd2/x              0.0470
test/moran/corr          0.8265
test/moran/gen_mean      0.4488
test/moran/mae           0.1417
test/moran/real_mean     0.5114
test/ot_w1/pos           0.2954
test/ot_w1/x             4.4157
test/ot_w2/pos           0.3500
test/ot_w2/x          

<frozen runpy>:128: RuntimeWarning: 'paired_slides_eval.evaluate' found in sys.modules after import of package 'paired_slides_eval', but prior to execution of 'paired_slides_eval.evaluate'; this may result in unpredictable behaviour
/work/magroup/shared/enfuzers/generative_modeling/eval_metrics/.venv/lib/python3.12/site-packages/docrep/decorators.py:43: SyntaxWarning: 'n_jobs' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)
/work/magroup/shared/enfuzers/generative_modeling/eval_metrics/.venv/lib/python3.12/site-packages/docrep/decorators.py:43: SyntaxWarning: 'show_progress_bar' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)
/work/magroup/shared/enfuzers/generative_modeling/eval_metrics/src/paired_slides_eval/metrics/morans.py:43: FutureWarning: Calling `spatial_neighbors` is deprecated and will be removed in squidpy v1.9.0. Use `spatial_neighbors_knn`, `spatial_neighbors_radius`, `spatial_neighbors_delaunay`, `spatial_neighbor

INFO     Creating graph using `None` transform and `1` libraries.               


/work/magroup/shared/enfuzers/generative_modeling/eval_metrics/src/paired_slides_eval/metrics/morans.py:43: FutureWarning: Calling `spatial_neighbors` is deprecated and will be removed in squidpy v1.9.0. Use `spatial_neighbors_knn`, `spatial_neighbors_radius`, `spatial_neighbors_delaunay`, `spatial_neighbors_grid`, or `spatial_neighbors_from_builder` instead.
  sq.gr.spatial_neighbors(adata, coord_type="generic", n_neighs=n_neighs)


INFO     Creating graph using `None` transform and `1` libraries.               
target: 9962 cells, 8 features
generated: 513 niches x 68 points
test/c2st/acc            0.6253
test/c2st/auc            0.6622
test/c2st/gene_acc       0.6485
test/c2st/gene_auc       0.7001
test/c2st/nn             0.3042
test/c2st/nn_flip        0.0655
test/c2st/nn_real_ref    0.4672
test/c2st/nn_std         0.1587
test/c2st/pos_acc        0.5633
test/ct/acc              0.5575
test/ct/acc_gap          0.3847
test/ct/acc_gen          0.2963
test/ct/acc_real         0.6810
test/ct/f1               0.5273
test/ct/prop_jsd         0.0103
test/ct/prop_kl          0.0452
test/ct/prop_tv          0.1209
test/mmd2/pos            0.0381
test/mmd2/x              0.0140
test/moran/corr          0.9919
test/moran/gen_mean      0.5210
test/moran/mae           0.0305
test/moran/real_mean     0.5114
test/ot_w1/pos           0.2703
test/ot_w1/x             1.4962
test/ot_w2/pos           0.3314
test/ot_w2/x          

**Optional** — collate the per-model `metrics.csv` files the CLI just wrote into one wide
comparison table (`reports/model_comparison.csv`)

In [6]:
import glob
import os

import pandas as pd

cols = {}
for csv_path in sorted(glob.glob("../reports/*/metrics_npc8.csv")):
    name = os.path.basename(os.path.dirname(csv_path))
    cols[name] = pd.read_csv(csv_path, index_col="metric")["value"]
table = pd.DataFrame(cols).sort_index()
table.to_csv("../reports/model_comparison_npc8.csv")
print("wrote ../reports/model_comparison_npc8.csv")
table

wrote ../reports/model_comparison_knn.csv


,nicheflow_cfm_unaligned,nicheflow_vfm,otcfm_1025,otcfm_spatial_1025
metric,,,,
test/c2st/acc,0.604500,0.589750,0.998500,0.998500
test/c2st/auc,0.642056,0.619103,0.999967,0.999931
test/c2st/nn,0.536100,0.370500,0.000000,0.000000
test/c2st/nn_flip,0.474000,0.165500,0.000000,0.000000
test/c2st/nn_real_ref,0.688450,0.460650,0.965350,0.962800
test/c2st/nn_std,0.161638,0.176804,0.000000,0.000000
test/c2st/pos_acc,0.590250,0.581250,0.631500,0.497750
test/ct/acc,0.395712,0.378168,0.384963,0.555712
test/ct/acc_gap,0.303450,0.307349,0.262355,0.068217


### 4b. A single model / subset of groups → print

Drop `--out` and the CLI just prints the metrics (plus the coord auto-decision under `notes:`). Swap
the `--generated` model and `--groups` to inspect a specific metric. Available groups: `psd`, `spd`,
`distribution` (MMD/EMD), `c2st`, `c2st_nn`, `moran`, and — with `--classifier` / `--regressor` —
`concordance`, `ct_gap`, `recon`.

In [ ]:
%%bash
cd ..
python -m paired_slides_eval.evaluate \
    --target data/abca_pair.pkl \
    --generated artifacts/otcfm_spatial_1025/generated.h5ad \
    --classifier ckpts/clf_spatial.ckpt --ct_real_reference fixed \
    --regressor ckpts/recon_spatial.ckpt \
    --ct_key class --groups distribution c2st c2st_nn moran recon